In [1]:
import pandas as pd

rounds = pd.read_csv("data/processed/rounds.csv", low_memory=False)
events = pd.read_csv("data/processed/events.csv")

# One row per player-event, round 1 only (guarantees every player appears once)
r1 = rounds[rounds['round_num'] == 1].copy()
r1 = r1.merge(events[['event_id_year', 'start_date']], on='event_id_year', how='left')
r1 = r1.sort_values(['dg_id', 'start_date'])

# Rolling means at player-event level
for n in [5, 10, 20]:
    r1[f'rolling_sg_{n}'] = (
        r1.groupby('dg_id')['sg_total']
        .transform(lambda x: x.shift(1).rolling(n, min_periods=3).mean())
    )

r1.to_csv("data/processed/rounds_features.csv", index=False)
